**0. Imports**

In [ ]:
import earthcarekit as eck
eck.set_config("/usr/people/raucher/Documents/Config_ECK/default_config.toml")
import xarray as xr
import numpy as np
from IPython.display import display
from pathlib import Path
from datetime import datetime, timedelta, date
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os, sys, shlex, shutil
from scipy.interpolate import interp1d

**0b. Remote ZIP Helpers**

In [ ]:
MY_SUBPROCESS_FILE = Path("/usr/people/raucher/Documents/Coding1/Gerd-Jan/OneDrive_1_24-2-2026")
if str(MY_SUBPROCESS_FILE) not in sys.path:
    sys.path.insert(0, str(MY_SUBPROCESS_FILE))
from my_subprocess import run_shell_cmd_and_communicate, print_shell_output

def _to_date(v):
    if isinstance(v, datetime): return v.date()
    if isinstance(v, date):     return v
    if isinstance(v, str):      return datetime.strptime(v, "%Y-%m-%d").date()
    raise TypeError(f"Unsupported type: {type(v)}")

def run_cmd_checked(cmd, verbose=False):
    lines_out, lines_err, rc = run_shell_cmd_and_communicate(cmd, verbose=verbose)
    if rc != 0:
        print_shell_output(lines_out, lines_err, prefix="[shell] ")
        raise RuntimeError(f"Command failed (exit {rc}): {cmd}")
    return lines_out, lines_err

def discover_remote_zip_files(remote_product_root, start_date, end_date):
    root, start, end = Path(remote_product_root), _to_date(start_date), _to_date(end_date)
    if end < start: raise ValueError("end_date must be >= start_date")
    zips, day = [], start
    while day <= end:
        d = root / day.strftime("%Y") / day.strftime("%m") / day.strftime("%d")
        if d.exists():
            zips.extend(sorted(d.glob("*.ZIP")))
            zips.extend(sorted(d.glob("*.zip")))
        day += timedelta(days=1)
    return sorted(dict.fromkeys(zips))

def stage_zip_and_extract(src_zip, local_stage_root):
    local_stage_root.mkdir(parents=True, exist_ok=True)
    local_zip   = local_stage_root / src_zip.name
    extract_dir = local_stage_root / src_zip.stem
    if local_zip.exists():   local_zip.unlink()
    if extract_dir.exists(): shutil.rmtree(extract_dir)
    run_cmd_checked(f"cp {shlex.quote(str(src_zip))} {shlex.quote(str(local_zip))}")
    run_cmd_checked(f"unzip -oq {shlex.quote(str(local_zip))} -d {shlex.quote(str(extract_dir))}")
    h5_files = sorted(extract_dir.rglob("*.h5"))
    if not h5_files: raise FileNotFoundError(f"No .h5 after extracting: {src_zip}")
    return local_zip, extract_dir, h5_files

def cleanup_staged_data(local_zip, extract_dir):
    if extract_dir is not None and extract_dir.exists(): shutil.rmtree(extract_dir)
    if local_zip   is not None and local_zip.exists():   local_zip.unlink()

**1. Config**

In [ ]:
REMOTE_PRODUCT_ROOT = Path("/net/pc230016/nobackup_1/users/zadelhof/EarthCARE_DATA/L2/ATL_TC__2A")

LOCAL_STAGE_ROOT = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/temp_ZIPs_TC_temp")
LOCAL_STAGE_ROOT.mkdir(parents=True, exist_ok=True)

# Full date range: October 2025 – February 2026
START_DATE = "2025-10-01"
END_DATE   = "2026-02-28"

OUTPUT_ROOT = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/ATC_output")

# Variables
TEMP_VAR   = "temperature"   # K
HEIGHT_VAR = "height"        # m
LAT        = "latitude"
LON        = "longitude"
QC_VAR     = "quality_status"
CLASS_VAR  = "classification"
MAX_QC_FLAG      = 1      # accept Good (0) and Likely Good (1)
EXCLUDE_CODES    = [-2, -1, -3]  # surface, noise, missing — skip these heights

GRID_RES_DEG         = 1.0
MAX_HEIGHT_M         = 20_000.0
MIN_SAMPLES_PER_CELL = 10

start_dt = datetime.strptime(START_DATE, "%Y-%m-%d").date()
end_dt   = datetime.strptime(END_DATE,   "%Y-%m-%d").date()
if not REMOTE_PRODUCT_ROOT.exists():
    raise FileNotFoundError(f"Remote path not mounted: {REMOTE_PRODUCT_ROOT}")

print(f"Product:    ATL_TC__2A  (temperature field, all valid pixels)")
print(f"Date range: {start_dt} to {end_dt}")
print(f"Grid:       {GRID_RES_DEG}°  |  max height: {MAX_HEIGHT_M:.0f} m")

**2. File Discovery**

In [ ]:
zip_paths = discover_remote_zip_files(REMOTE_PRODUCT_ROOT, start_dt, end_dt)
print(f"Discovered {len(zip_paths)} ZIP files  |  {start_dt} -> {end_dt}")
for p in zip_paths[:3]: print(" ", p)
if len(zip_paths) > 3: print(f"  ... and {len(zip_paths)-3} more")
if not zip_paths: raise FileNotFoundError("No ZIP files found.")

**3. Single-File Inspection**

In [ ]:
local_zip = extract_dir = None
try:
    local_zip, extract_dir, staged_h5 = stage_zip_and_extract(zip_paths[0], LOCAL_STAGE_ROOT)
    with eck.read_product(str(staged_h5[0])) as ds0:
        print("height shape:", ds0[HEIGHT_VAR].shape)
        display(ds0)
        if TEMP_VAR not in ds0.data_vars:
            print(f"\nWARNING: '{TEMP_VAR}' not found in this file baseline.")
            print("This likely means the local files are a different product baseline than the work PC.")
            print(f"Available vars: {[v for v in ds0.data_vars if not v.startswith('aerosol')]}")
        else:
            t = ds0[TEMP_VAR].values.astype(float)
            valid = np.isfinite(t)
            print(f"\n{TEMP_VAR}: min={np.nanmin(t):.1f} K  max={np.nanmax(t):.1f} K  "
                  f"valid={100*valid.mean():.1f}%  units={ds0[TEMP_VAR].attrs.get('units','?')}")
finally:
    cleanup_staged_data(local_zip, extract_dir)

**4. Grid Setup**

In [ ]:
lat_bins = np.arange(-90.0, 90.0  + GRID_RES_DEG, GRID_RES_DEG)
lon_bins = np.arange(-180.0, 180.0 + GRID_RES_DEG, GRID_RES_DEG)
lat_centers = 0.5 * (lat_bins[:-1] + lat_bins[1:])
lon_centers = 0.5 * (lon_bins[:-1] + lon_bins[1:])
n_lat, n_lon = len(lat_centers), len(lon_centers)

target_h = np.arange(0.0, MAX_HEIGHT_M + 1.0, 100.0)   # 201 levels: 0..20 000 m
n_height  = target_h.size

# Global accumulators
temp_sum_3d   = np.zeros((n_lat, n_lon, n_height), dtype=np.float64)
temp_count_3d = np.zeros((n_lat, n_lon, n_height), dtype=np.float64)

print(f"Grid shape (lat, lon, height): {(n_lat, n_lon, n_height)}")
print(f"Height range: {float(target_h[0]):.0f} .. {float(target_h[-1]):.0f} m")

**5. One-File Accumulator**

In [ ]:
"""
For each ATL_TC__2A file:
  - Keep pixels where height and temperature are finite, classification not in
    EXCLUDE_CODES, and quality_status <= MAX_QC_FLAG.
  - Interpolate temperature linearly onto target_h per profile.
  - Accumulate temp_sum and temp_count on the (lat, lon, height) grid
    using np.bincount for speed.
  - Files without a 'temperature' variable are skipped with a warning
    (older baselines may lack this field).
"""

def accumulate_one_file(fp, lat_bins, lon_bins, target_h):
    n_lb, n_lonb, n_h = len(lat_bins)-1, len(lon_bins)-1, target_h.size
    ts = np.zeros((n_lb, n_lonb, n_h), dtype=np.float64)
    tc = np.zeros((n_lb, n_lonb, n_h), dtype=np.float64)
    flat_size = n_lb * n_lonb * n_h

    with eck.read_product(str(fp)) as ds:
        if TEMP_VAR not in ds.data_vars:
            print(f"  SKIP (no '{TEMP_VAR}'): {Path(fp).name}")
            return ts, tc
        h    = ds[HEIGHT_VAR].values.astype(float)
        lat  = ds[LAT].values
        lon  = ds[LON].values
        temp = ds[TEMP_VAR].values.astype(float)
        cls  = ds[CLASS_VAR].values.astype(float)
        qc   = ds[QC_VAR].values.astype(int)
        n_obs = h.shape[0]

    temp_interp = np.full((n_obs, n_h), np.nan, dtype=np.float64)

    for i in range(n_obs):
        h_i, t_i, cls_i, qc_i = h[i], temp[i], cls[i], qc[i]
        valid = (np.isfinite(h_i) & np.isfinite(t_i)
                 & ~np.isin(cls_i, EXCLUDE_CODES)
                 & (qc_i <= MAX_QC_FLAG))
        if np.sum(valid) < 2:
            continue
        h_v = h_i[valid]; t_v = t_i[valid]
        order = np.argsort(h_v)
        h_v, t_v = h_v[order], t_v[order]
        h_v, idx = np.unique(h_v, return_index=True)
        t_v = t_v[idx]
        if h_v.size < 2:
            continue
        temp_interp[i] = interp1d(h_v, t_v, kind="linear",
                                   bounds_error=False, fill_value=np.nan)(target_h)

    # Vectorised binning
    lat_idx = np.searchsorted(lat_bins[1:-1], lat)
    lon_idx = np.searchsorted(lon_bins[1:-1], lon)
    valid_ll = (lat_idx < n_lb) & (lon_idx < n_lonb)

    lat_idx_2d = np.broadcast_to(lat_idx[:, None], (n_obs, n_h))
    lon_idx_2d = np.broadcast_to(lon_idx[:, None], (n_obs, n_h))
    h_idx_2d   = np.broadcast_to(np.arange(n_h)[None, :], (n_obs, n_h))
    valid_base = np.broadcast_to(valid_ll[:, None], (n_obs, n_h))

    m = valid_base & np.isfinite(temp_interp)
    if m.any():
        li = lat_idx_2d[m].astype(np.intp)
        lo = lon_idx_2d[m].astype(np.intp)
        hi = h_idx_2d[m].astype(np.intp)
        tv = temp_interp[m]
        fi = li * (n_lonb * n_h) + lo * n_h + hi
        tc += np.bincount(fi, minlength=flat_size).reshape(n_lb, n_lonb, n_h)
        ts += np.bincount(fi, weights=tv, minlength=flat_size).reshape(n_lb, n_lonb, n_h)

    return ts, tc


### TEST on first file ###
local_zip = extract_dir = None
try:
    local_zip, extract_dir, staged_h5 = stage_zip_and_extract(zip_paths[0], LOCAL_STAGE_ROOT)
    ts1, tc1 = accumulate_one_file(staged_h5[0], lat_bins, lon_bins, target_h)
    if np.nansum(tc1) > 0:
        mean_t = np.divide(ts1, tc1, out=np.full_like(ts1, np.nan), where=tc1>0)
        print(f"One-file test: {int(np.nansum(tc1))} valid pixels")
        print(f"  temperature: mean={np.nanmean(mean_t):.1f} K  "
              f"min={np.nanmin(mean_t):.1f}  max={np.nanmax(mean_t):.1f}")
    else:
        print("No temperature data in test file (baseline mismatch — expected on work PC files)")
finally:
    cleanup_staged_data(local_zip, extract_dir)

**6. Few-File Test**

In [ ]:
N_QUICK = min(5, len(zip_paths))
test_ts = np.zeros_like(temp_sum_3d)
test_tc = np.zeros_like(temp_count_3d)
failed_test = []

for src_zip in zip_paths[:N_QUICK]:
    local_zip = extract_dir = None
    try:
        local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_ROOT)
        for fp in h5_files:
            ts, tc = accumulate_one_file(fp, lat_bins, lon_bins, target_h)
            test_ts += ts; test_tc += tc
    except Exception as ex:
        failed_test.append(str(ex))
    finally:
        cleanup_staged_data(local_zip, extract_dir)

mean_t = np.divide(test_ts, test_tc, out=np.full_like(test_ts, np.nan), where=test_tc>0)
print(f"Test ({N_QUICK} ZIPs): {int(np.nansum(test_tc))} pixels | failures={len(failed_test)}")
print(f"Temperature: {np.nanmean(mean_t):.1f} K mean  |  valid cells: {np.isfinite(mean_t).sum()}")
if failed_test: print("First failure:", failed_test[0])

**7. Full Batch Processing**

In [ ]:
# Re-initialise (safe to re-run)
temp_sum_3d   = np.zeros((n_lat, n_lon, n_height), dtype=np.float64)
temp_count_3d = np.zeros((n_lat, n_lon, n_height), dtype=np.float64)

failed_zips  = []
processed_h5 = 0
n_zips       = len(zip_paths)

for idx, src_zip in enumerate(zip_paths, start=1):
    local_zip = extract_dir = None
    try:
        local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_ROOT)
        for fp in h5_files:
            ts, tc = accumulate_one_file(fp, lat_bins, lon_bins, target_h)
            temp_sum_3d   += ts
            temp_count_3d += tc
            processed_h5  += 1
    except Exception as e:
        failed_zips.append((str(src_zip), str(e)))
    finally:
        cleanup_staged_data(local_zip, extract_dir)
    if idx % 20 == 0 or idx == n_zips:
        print(f"{idx}/{n_zips} ZIPs | h5={processed_h5} | failed={len(failed_zips)}")

print("\nBatch done")
print(f"H5 files: {processed_h5}  |  ZIP failures: {len(failed_zips)}")
if failed_zips:
    print("First failure:", failed_zips[0][1])

**8. Compute Mean Temperature**

In [ ]:
# 3-D mean temperature: (lat, lon, height)
temp_mean_3d = np.divide(
    temp_sum_3d, temp_count_3d,
    out=np.full_like(temp_sum_3d, np.nan),
    where=temp_count_3d >= MIN_SAMPLES_PER_CELL,
)

# --- World map: column mean (average over height axis) ---
# Weight each height level by the number of samples so sparsely sampled
# altitudes don't bias the average.
ts_col  = np.nansum(temp_sum_3d,   axis=2)   # (n_lat, n_lon)
tc_col  = np.nansum(temp_count_3d, axis=2)
temp_map_2d = np.divide(
    ts_col, tc_col,
    out=np.full_like(ts_col, np.nan),
    where=tc_col >= MIN_SAMPLES_PER_CELL,
)

# --- Lat/height: zonal mean (average over longitude axis) ---
ts_lh  = np.nansum(temp_sum_3d,   axis=1)   # (n_lat, n_height)
tc_lh  = np.nansum(temp_count_3d, axis=1)
temp_mean_lh = np.divide(
    ts_lh, tc_lh,
    out=np.full_like(ts_lh, np.nan),
    where=tc_lh >= MIN_SAMPLES_PER_CELL,
)

print("Statistics computed")
print(f"World map  (lat x lon):    valid cells = {np.isfinite(temp_map_2d).sum()}  "
      f"mean = {np.nanmean(temp_map_2d):.1f} K")
print(f"Lat/height (lat x height): valid cells = {np.isfinite(temp_mean_lh).sum()}  "
      f"mean = {np.nanmean(temp_mean_lh):.1f} K")

**9. Save to NetCDF**

In [ ]:
outdir = f"{OUTPUT_ROOT}/{start_dt:%Y%m%d}_{end_dt:%Y%m%d}"
os.makedirs(outdir, exist_ok=True)
_base = f"ATL_TC_2A_temperature_{GRID_RES_DEG}deg_{start_dt:%Y%m%d}_{end_dt:%Y%m%d}"

xr.Dataset(
    {"temperature": (["latitude", "longitude", "height"], temp_mean_3d),
     "temp_count":  (["latitude", "longitude", "height"], temp_count_3d)},
    coords={"latitude": lat_centers, "longitude": lon_centers, "height": target_h},
    attrs={"title": "ATL_TC__2A mean temperature",
           "start": str(start_dt), "end": str(end_dt)},
).to_netcdf(f"{outdir}/{_base}_3d.nc")

xr.Dataset(
    {"temperature": (["latitude", "height"], temp_mean_lh),
     "temp_count":  (["latitude", "height"], tc_lh)},
    coords={"latitude": lat_centers, "height": target_h},
).to_netcdf(f"{outdir}/{_base}_latheight.nc")

print(f"Saved to {outdir}/")
print(f"  {_base}_3d.nc")
print(f"  {_base}_latheight.nc")

**10. Plot: Temperature World Map (height-integrated)**

In [ ]:
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print("cartopy not available — falling back to plain imshow")

vmin_map = np.nanpercentile(temp_map_2d, 2)
vmax_map = np.nanpercentile(temp_map_2d, 98)

if HAS_CARTOPY:
    fig = plt.figure(figsize=(14, 6))
    ax  = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
    ax.set_global()
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color="k")
    ax.add_feature(cfeature.BORDERS,   linewidth=0.3, color="grey")
    im = ax.pcolormesh(
        lon_centers, lat_centers, temp_map_2d,
        transform=ccrs.PlateCarree(),
        cmap="RdYlBu_r", vmin=vmin_map, vmax=vmax_map,
    )
else:
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(
        temp_map_2d, origin="lower", aspect="auto",
        extent=[-180, 180, -90, 90],
        cmap="RdYlBu_r", vmin=vmin_map, vmax=vmax_map,
    )
    ax.set_xlabel("Longitude (°)")
    ax.set_ylabel("Latitude (°)")

cbar = plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.05, fraction=0.04)
cbar.set_label("Mean temperature (K)")
cbar.ax.text(0.5, -2.2,
             f"Data min = {np.nanmin(temp_map_2d):.1f},  Max = {np.nanmax(temp_map_2d):.1f}",
             ha="center", va="top", transform=cbar.ax.transAxes, fontsize=8)

ax.set_title(f"ATL TC mean temperature (height-integrated)\n"
             f"{start_dt:%Y-%m-%d} to {end_dt:%Y-%m-%d}", fontsize=12)

plt.tight_layout()
fig.savefig(f"{outdir}/{_base}_worldmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("World map saved.")

**11. Plot: Temperature Lat/Height (longitude-integrated)**

In [ ]:
# Mask to displayed height range (0–20 km)
h_mask = target_h <= MAX_HEIGHT_M
plot_h  = target_h[h_mask] / 1000.0          # metres -> km for y-axis
plot_lh = temp_mean_lh[:, h_mask]            # (n_lat, n_h_shown)

vmin_lh = np.nanpercentile(plot_lh, 2)
vmax_lh = np.nanpercentile(plot_lh, 98)

fig, ax = plt.subplots(figsize=(10, 6))

im = ax.pcolormesh(
    lat_centers, plot_h, plot_lh.T,
    cmap="RdYlBu_r", vmin=vmin_lh, vmax=vmax_lh,
    shading="nearest",
)

cbar = plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.12, fraction=0.04)
cbar.set_label("atl tc temperature (K)")
# Add data range note below colorbar (matching screenshot style)
cbar.ax.text(0.5, -2.5,
             f"Data Min = {np.nanmin(plot_lh):.1f},  Max = {np.nanmax(plot_lh):.1f}",
             ha="center", va="top", transform=cbar.ax.transAxes, fontsize=8)

ax.set_xlabel("latitude", fontsize=11)
ax.set_ylabel("height (km)", fontsize=11)
ax.set_xlim(-90, 90)
ax.set_ylim(plot_h[0], plot_h[-1])
ax.set_xticks(np.arange(-90, 91, 18))
ax.set_title(f"atl tc temperature\n{start_dt:%Y-%m-%d} to {end_dt:%Y-%m-%d}", fontsize=12)
ax.set_facecolor("lightgrey")   # grey = no data, matching screenshot

plt.tight_layout()
fig.savefig(f"{outdir}/{_base}_latheight.png", dpi=150, bbox_inches="tight")
plt.show()
print("Lat/height plot saved.")